In [2]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [4]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [5]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [6]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.9560439560439561,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.9159951159951161,
  'support': 300.0}}

In [7]:
import mlflow
import mlflow.sklearn

In [8]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("First Experiment")

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metrics({
        'accuracy': report_dict['accuracy'],
        'recall_class_0': report_dict['0']['recall'],
        'recall_class_1': report_dict['1']['recall'],
        'f1_score_macro': report_dict['macro avg']['f1-score']
    })
    mlflow.sklearn.log_model(
        sk_model=lr, 
        name="Logistic Regression"
    ) 

2026/04/21 17:58:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run stylish-donkey-368 at: http://127.0.0.1:5000/#/experiments/122792658621644364/runs/aab6e7b273ff4231a96a16c2823b0f45
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/122792658621644364


In [9]:
import os
import shutil
import pandas as pd
import mlflow.sklearn

local_model_path = "first_experiment_model"

if os.path.exists(local_model_path):
    shutil.rmtree(local_model_path)

input_example = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]])

mlflow.sklearn.save_model(
    sk_model=lr,
    path=local_model_path,
    input_example=input_example
)

print("Saved at:", os.path.abspath(local_model_path))
print("Files:", os.listdir(local_model_path))

2026/04/21 17:58:17 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: ValueError('X has 4 features, but LogisticRegression is expecting 10 features as input.'). To see the full traceback, set the logging level to DEBUG via `logging.getLogger("mlflow").setLevel(logging.DEBUG)`.


Saved at: c:\Users\aag52\mlflow_dagshub_demo\first_experiment_model
Files: ['conda.yaml', 'input_example.json', 'MLmodel', 'model.pkl', 'python_env.yaml', 'requirements.txt', 'serving_input_example.json']


In [10]:
print(type(X_test))

<class 'numpy.ndarray'>


In [11]:
print(X_test[0].tolist())

[0.1276369210104354, 0.5388543360516529, -0.677549384520644, -0.5306479123499234, 0.2697207514136527, 0.286215260265736, 0.24551629442856485, -0.17119612639661974, -1.0487003944182158, 0.19594920307479496]
